In [1]:
# отбираем данные для дообучения yandexgpt размером 50 тыс. строк и для теста 900 строк с равным количеством строк каждого класса (по 100)
# для консистентности выбрем те же самые строки, что и в предыдущих версиях

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option("display.max_colwidth", None)

data_download = pd.read_parquet("../../uc_data_labeling/data_download_09102025_uc.parquet")


train_index = pd.read_csv("X_train_id.csv",index_col=0)
test_index = pd.read_csv("X_test_balanced_id.csv",index_col=0)

In [3]:
# отбираем нужные строки для train, 
# несколько строк поменяли id, поэтому их отберем по тексту назначения
ids = train_index['id'][train_index['id'].isin(data_download['id'])]
train = data_download.set_index('id').loc[ids].reset_index()

diff = train_index[~train_index['id'].isin(train['id'])]
dd_diff = data_download[data_download['purpose'].isin(diff['purpose'])]

train = pd.concat([train,dd_diff],ignore_index=True)

# сделаем проверку по текстам назначений
display(train.id.count(),
        set(train.purpose) - set(train_index.purpose),
        set(train_index.purpose) - set(train.purpose))

49999

set()

set()

In [4]:
# отбираем нужные строки для test
ids = test_index['id'][test_index['id'].isin(data_download['id'])]
test = data_download.set_index('id').loc[ids].reset_index()

# сделаем проверку по текстам назначений
display(test.id.count(),
        set(test.purpose) - set(test_index.purpose),
        set(test_index.purpose) - set(test.purpose))

900

set()

set()

In [5]:
# проверим пропуски по основным признакам
display(train[['purpose','articles__name','projects__name','counterparties__name','universal_category']].isna().sum(),
        test[['purpose','articles__name','projects__name','counterparties__name','universal_category']].isna().sum())
# и заполним в них пропуски

train[['purpose','articles__name','projects__name','counterparties__name','universal_category']] = train[['purpose','articles__name','projects__name','counterparties__name','universal_category']].fillna('отсутствует') 
test[['purpose','articles__name','projects__name','counterparties__name','universal_category']] = test[['purpose','articles__name','projects__name','counterparties__name','universal_category']].fillna('отсутствует')


purpose                    0
articles__name             0
projects__name          1955
counterparties__name    4249
universal_category         0
dtype: int64

purpose                  0
articles__name           0
projects__name          67
counterparties__name    89
universal_category       0
dtype: int64

In [6]:
# переформатируем датасеты под дообучение yandexgpt

train.loc[:, 'text']  = train.apply(
    lambda x: (
        f"""назначение платежа: {x['purpose']}
название статьи: {x['articles__name']}
название проекта: {x['projects__name']}
категория донора: {x['counterparties__name']}"""
    ),
    axis=1
)

test.loc[:, 'text']  = test.apply(
    lambda x: (
        f"""назначение платежа: {x['purpose']}
название статьи: {x['articles__name']}
название проекта: {x['projects__name']}
категория донора: {x['counterparties__name']}"""
    ),
    axis=1
)

# и разделим признаки

X_train = train['text']
y_train = train['universal_category']

X_test_balanced = test['text']
y_test_balanced = test['universal_category']

In [7]:
print(X_train.head(3),
      y_train.head(3),
      X_test_balanced.head(3),
      y_test_balanced.head(3))

0    назначение платежа: Перечисление денежных средств по договору №ПД-1412, по платежу за 14.02.2024; ID платежа 13322271632. НДС не облагается\nназвание статьи: Юр.лица\nназвание проекта: Травли NET\nкатегория донора: ?_нужно_проставить_донора
1                                              назначение платежа: Благотворительное пожертвование на уставную деятельность. НДС не облагается\nназвание статьи: Тинькофф приложение\nназвание проекта: Уставные цели\nкатегория донора: ... массовые
2                                              назначение платежа: Благотворительное пожертвование на уставную деятельность. НДС не облагается\nназвание статьи: Тинькофф приложение\nназвание проекта: Уставные цели\nкатегория донора: ... массовые
Name: text, dtype: object 0    пожертвования от юридических лиц (напрямую)
1     пожертвования от физических лиц (напрямую)
2     пожертвования от физических лиц (напрямую)
Name: universal_category, dtype: object 0    назначение платежа: Перечисление средств по 

In [8]:
# сохраняем это все добро
X_train.to_csv('X_train.csv')
y_train.to_csv('y_train.csv')
X_test_balanced.to_csv('X_test_balanced.csv')
y_test_balanced.to_csv('y_test_balanced.csv')


In [9]:
# для обучения yandexgpt сохраним в json
train_json = pd.DataFrame({
    "text": X_train,
    "label": y_train
})

train_json = pd.concat([train_json["text"], pd.get_dummies(train_json["label"]).astype(int)], axis=1)
train_json.to_json("train.json", orient="records", lines=True, force_ascii=False)